# Cuvis Python SDK Example 3
## Load and reprocess a recorded measurement

In this example an already recorded measurement (SessionFile .cu3s) is loaded.
The measurement is reprocessed into different processing modes, explaining their differences.

**Used principles:**
 - *SessionFile* to load a recorded measurement
 - *Measurement* to access the SessionFiles data and meta-data
 - *ProcessingContext* to generate hyperspectral cubes using different processing modes

**Step-by-Step overview for this example:**
 1. Import and initialize Cuvis SDK
 2. Load the recorded measurement file using *SessionFile*
 3. Extract a *Measurement* from the *SessionFile*
 4. Initialize a *ProcessingContext* from a *SessionFile*
 5. Use *ProcessingContext* to generate hyperspectral cubes

**Prerequisites to running this example:**
 - Have a recorded measurement in *SessionFile* format (.cu3s)
 - Have a recorded White and Dark reference measurement
 - Have the Cuvis SDK installed
 - Have Python and the and the Python packages **cuvis** and **notebook** installed

In [ ]:
# If the import of cuvis fails, the most common cause is a mismatch between
# the _cuvis_ python package and the installed version of the Cuvis SDK.
# Try re-installing both and make sure that the version numbers match exactly
import cuvis
import time
from matplotlib import pyplot as plt
import numpy as np
print("Cuvis Python SDK Example 3")

# Initialize the Cuvis SDK using a settings-directory
# This is optional (all settings have defaults),
# but enables you to optimize Cuvis' performance on your system using the settings
# Your camera and the default Cuvis installation both provide these settings files
print("Initializing Cuvis")
cuvis.General.init("./settings")

In [ ]:
# Enter paths applicable to your setup here
session_file_path = "D:/TestMesus/Blumen_NOPAN/Auto_001.cu3s"
dark_reference_file_path = "D:/TestMesus/Blumen_NOPAN/Auto_001.cu3s"
white_reference_file_path = "D:/TestMesus/Blumen_NOPAN/Auto_001.cu3s"

# Load the SessionFile
print(F"Loading Session File '{session_file_path}'")
session = cuvis.SessionFile(session_file_path)
# Fetch the first measurement from the session file
measurement = session.get_measurement(0)

# Load dark reference measurement
print(F"Loading Dark reference file '{dark_reference_file_path}'")
dark_sess = cuvis.SessionFile(dark_reference_file_path)
if (dark_mesu := dark_sess.get_reference(0, cuvis.ReferenceType.Dark)) is not None:
    print("Using Dark reference from Session File") 
else:
    print("Using first measurement as Dark reference") 
    dark_mesu = dark_sess.get_measurement(0)

# Load white reference measurement
print(F"Loading White reference file '{white_reference_file_path}'")
white_sess = cuvis.SessionFile(white_reference_file_path)
if (white_mesu := white_sess.get_reference(0, cuvis.ReferenceType.White)) is not None:
    print("Using White reference from Session File") 
else:
    print("Using first measurement as White reference") 
    white_mesu = white_sess.get_measurement(0)

In [ ]:
# Helper functions

# Read and print processing mode of the measurement
def print_processing_mode(measurement):
    procmode = measurement.processing_mode
    if procmode == cuvis.ProcessingMode.Preview:
        print("Measurement does not have a hyperspectral cube: Preview Mode")
    elif procmode == cuvis.ProcessingMode.Raw:
        print("Measurement has a hyperspectral cube with raw counts: Raw Mode")
    elif procmode == cuvis.ProcessingMode.DarkSubtract:
        print("Measurement has a hyperspectral cube with raw counts: Dark Subtract Mode")
    elif procmode == cuvis.ProcessingMode.Reflectance:
        print("Measurement has a hyperspectral cube with reflectance values: Reflectance Mode")
    elif procmode == cuvis.ProcessingMode.SpectralRadiance:
        print("Measurement has a hyperspectral cube with Spectral Radiance values: Spectral Radiance Mode")
    else:
        print("Unknown processing mode")

def show_spectrum_and_channel(measurement):
    cube = measurement.cube
    x = cube.width // 2
    y = cube.height // 2
    c = cube.channels // 2

    procmode = measurement.processing_mode
    if procmode == cuvis.ProcessingMode.Preview:
        print("Measurement does not have a hyperspectral cube: Preview Mode")
        return
    elif procmode in (cuvis.ProcessingMode.Raw, cuvis.ProcessingMode.DarkSubtract):
        ylabel = "Counts"
    elif procmode == cuvis.ProcessingMode.Reflectance:
        ylabel = "Reflectance [%]"
    elif procmode == cuvis.ProcessingMode.SpectralRadiance:
        ylabel = "Spectral Radiance [W / m² / sr / µm]"
    else:
        print("Unknown processing mode")
        return
    
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

    # Spectrum at (x, y)
    spectrum = cube.array[y,x,:]
    if procmode == cuvis.ProcessingMode.Reflectance:
        spectrum = np.array(spectrum, dtype=np.float32)
        spectrum /= 100
    ax0.plot(cube.wavelength, spectrum)
    ax0.set_xlabel("Wavelength")
    ax0.set_ylabel(ylabel)
    ax0.set_title(F"Spectrum at (x={x}, y={y})")
    ax0.grid(True, alpha=0.3)

    # Single channel image
    channel_image = cube.array[:,:,c]
    im = ax1.imshow(channel_image, cmap="gray")
    ax1.set_title(F"Channel {c}")
    fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

    return fig, (ax0, ax1)

In [ ]:
print_processing_mode(measurement)

#### Processing Context
The *ProcessingContext* is the interface that enables computing a hyperspectral cube from a measurement.
A camera calibration file is required to initialize the *ProcessingContext*, as each Cubert camera is individually calibrated to provide the most accurate spectral information.
As a SessionFile contains the camera calibration, it is used to construct the *ProcessingContext*.

To generate a hyperspectral cube, the *ProcessingContext* is **applied** to the *Measurement*. The *Measurement* is modified **in-place** and now contains a cube.

To select the processing mode, write the **processing_mode** attribute.

When initializing a *ProcessingContext* from a *SessionFile*, the reference *Measurements* stored in the *SessionFile* are automatically loaded and set within the *ProcessingContext*.
Using the method *set_reference*, different measurements can be set for each reference type.

In [ ]:
processing_context = cuvis.ProcessingContext(session)

#### Processing Mode RAW
The mode *RAW* is the most basic processing applied to generate a hyperspectral cube.
No references are necessary for this mode; it also provides the least refined data.
The data is measured in sensor counts, no physical unit can be assigned to this.
The cube data is provided as an integer format.
Either unsigned 8 bit, if the sensor pixel format is Mono8, else unsigned 16 bit.

In [ ]:
# Set processing mode
processing_context.processing_mode = cuvis.ProcessingMode.Raw

# Compute a cube in Raw mode for the measurement
processing_context.apply(measurement)
# Show data
print_processing_mode(measurement)
show_spectrum_and_channel(measurement)

#### Processing Mode Dark Subtract
The mode *Dark Subtract* is similar to *RAW*, as it also measures the data in sensor counts.
A Dark reference is used in the computation for this mode to mitigate the effect of inherent sensor noise, refining the data.

In [ ]:
# Set processing mode
processing_context.processing_mode = cuvis.ProcessingMode.DarkSubtract

# Set dark reference
processing_context.set_reference(dark_mesu, cuvis.ReferenceType.Dark)

# Compute a cube in DarkSubtract mode for the measurement
processing_context.apply(measurement)
# Show data
print_processing_mode(measurement)
show_spectrum_and_channel(measurement)

#### Processing Mode Reflectance
The mode *Reflectance* changes how the cube is generated and its data type.
The cube now provides data in the form of reflected light as a percentage.
A Dark and a White reference are used to compute this percentage, ie. 100% reflectance means that the data at this point is as bright as the White reference.

*Please note:*
The cube data are still provided in an integer format (unsigned 16 bit) to speed up processing and reduce storage.
The fixed-point format is defined as such:
 - 0 = 0% Reflectance (as bright as Dark reference)
 - 10000 = 100% Reflectance (as bright as White reference)
 - 30000 = 300% Reflectance (3 times as bright as White reference)
 - 60000 = 600% Reflectance (6 times as bright as White reference)
 - etc.

In [ ]:
# Set processing mode
processing_context.processing_mode = cuvis.ProcessingMode.Reflectance

# Set references
processing_context.set_reference(dark_mesu, cuvis.ReferenceType.Dark)
processing_context.set_reference(white_mesu, cuvis.ReferenceType.White)

# Compute a cube in Reflectance mode for the measurement
processing_context.apply(measurement)
# Show data
print_processing_mode(measurement)
show_spectrum_and_channel(measurement)

#### Processing Mode Spectral Radiance
The mode *Spectral Radiance* changes how the cube is generated and its data type.
The cube now provides data in the form of a physical unit: W / m² / sr / µm

A Dark reference and a special Spectral Radiance reference are used to compute this.

The Spectral Radiance reference is created by default by Cubert using calibrated measurement equipment during camera build-up and calibration.
It is contained in the camera calibration file.
For older cameras it used to be provided as the file *SpRad.cu3* and may not have been delivered for your camera.

The cube data are provided in a floating point format (float 32 bit).

In [ ]:
# Set processing mode
processing_context.processing_mode = cuvis.ProcessingMode.SpectralRadiance

# Set dark reference
processing_context.set_reference(dark_mesu, cuvis.ReferenceType.Dark)

# Compute a cube in Spectral Radiance mode for the measurement
processing_context.apply(measurement)
# Show data
print_processing_mode(measurement)
show_spectrum_and_channel(measurement)

#### Distance Calibration
Most Ultris cameras (except for Relay-variants) require distance calibration to achieve optimal results.
Distance calibration is an operation that can be done with already recorded data and requires a distance reference measurement.
The reference should contain high-contrast data over the relevant spectral channels at the desired distance that data should be calibrated to.

In this example, the measurement itself will be used as the distance reference. If the target object is suitable (high contrast, non-repeating patterns), this can suffice for good results.

In [ ]:
# Set processing mode
processing_context.processing_mode = cuvis.ProcessingMode.Reflectance

# Set references
processing_context.set_reference(dark_mesu, cuvis.ReferenceType.Dark)
processing_context.set_reference(white_mesu, cuvis.ReferenceType.White)
processing_context.set_reference(measurement, cuvis.ReferenceType.Distance)

# Compute a cube in Spectral Radiance mode for the measurement
processing_context.apply(measurement)
# Show data
print_processing_mode(measurement)
show_spectrum_and_channel(measurement)